# Static FEA vs. Dynamic FEA Significance Tests

This notebook compares the static and dynamic FEA-based FER models on the same held-out test reenactments.

Both prediction CSV files contain two view-level rows per reenactment, but the FEA prediction is identical for Central and Side within a reenactment. The analysis therefore collapses each file to exactly **one paired FEA correctness value per reenactment**, yielding **378 paired observations**.

## Analysis plan

1. Primary comparison: use an **exact two-sided McNemar test** on paired Static- and Dynamic-FEA correctness over the same 378 reenactments.
2. Effect-size uncertainty: estimate a **paired reenactment bootstrap 95% confidence interval** for the accuracy difference $\Delta = \mathrm{Accuracy}_{Dynamic\ FEA} - \mathrm{Accuracy}_{Static\ FEA}$.
3. Sensitivity check: test the same 378 paired differences with a one-sample t-test against zero.
4. Participant-level heterogeneity check: summarize the Static-vs.-Dynamic FEA difference separately for each of the eight held-out participants.

### Why McNemar is the primary test

Static and Dynamic FEA each contribute one binary correctness value $\{0,1\}$ for the same reenactment. McNemar's test is therefore the natural paired test because it evaluates whether the two models differ in their discordant outcomes while preserving the reenactment pairing.

The test is two-sided. Positive accuracy differences are interpreted as favoring Dynamic FEA, but the inferential test itself does not assume a direction in advance.

The participant-level analysis is descriptive only. The primary inference concerns the fixed held-out test set and its reenactments rather than population-level generalization across participants.

## 1. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import ttest_1samp
from statsmodels.stats.contingency_tables import mcnemar


STATIC_PREDICTIONS_PATH = Path("static_test_predictions.csv")
DYNAMIC_PREDICTIONS_PATH = Path("dynamic_test_predictions.csv")

RANDOM_SEED = 42

# Use many resamples for a stable confidence interval.
# Resampling is processed in batches below to keep memory usage low.
N_BOOTSTRAP = 1_000_000
BOOTSTRAP_BATCH_SIZE = 10_000

ALPHA = 0.05

## 2. Load and Validate Predictions

Both prediction CSV files are expected to contain one row per image-view sample. For this comparison, only the FEA prediction is used.

Each file must contain:

- `sample_id`
- `reenactment_id`
- `participant_id`
- `camera_index`
- `true_label_id`
- `fea_pred_id`

The Static and Dynamic files must contain exactly the same 378 reenactments and matching ground-truth metadata.

In [2]:
static_prediction_df = pd.read_csv(STATIC_PREDICTIONS_PATH)
dynamic_prediction_df = pd.read_csv(DYNAMIC_PREDICTIONS_PATH)

print(f"Static rows:  {len(static_prediction_df)}")
print(f"Dynamic rows: {len(dynamic_prediction_df)}")
print(f"Static columns:  {len(static_prediction_df.columns)}")
print(f"Dynamic columns: {len(dynamic_prediction_df.columns)}")

static_prediction_df.head()

Static rows:  756
Dynamic rows: 756
Static columns:  39
Dynamic columns: 39


,sample_id,reenactment_id,timestamp,set_id,participant_id,level_id,emoji_id,camera_index,perspective,true_label_id,...,fea_prob_surprise,multimodal_pred_id,multimodal_pred,multimodal_prob_anger,multimodal_prob_disgust,multimodal_prob_fear,multimodal_prob_happiness,multimodal_prob_neutral,multimodal_prob_sadness,multimodal_prob_surprise
0,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Central,0,...,0.004558,0,Anger,0.677313,0.030212,0.002842,0.003746,0.026476,0.252719,0.006691
1,1700478995850-2-1-1-0-0-1,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,1,Side,0,...,0.004558,4,Neutral,0.095327,0.056766,0.039777,0.011597,0.752703,0.013406,0.030424
2,1700478998549-2-1-1-1-5-0,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,0,Central,5,...,0.000017,5,Sadness,0.000005,0.000117,0.000001,0.000006,0.000001,0.999867,0.000003
3,1700478998549-2-1-1-1-5-1,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,1,Side,5,...,0.000017,5,Sadness,0.000149,0.000589,0.000038,0.000047,0.000028,0.999063,0.000088
4,1700479001137-2-1-1-2-3-0,1700479001137-2-1-1-2-3,1700479001137,2,1,1,2,0,Central,3,...,0.000797,3,Happiness,0.000376,0.001057,0.000616,0.995559,0.000666,0.000936,0.000790


In [3]:
required_columns = {
    "sample_id",
    "reenactment_id",
    "participant_id",
    "camera_index",
    "true_label_id",
    "fea_pred_id"
}

for name, df in [("Static", static_prediction_df), ("Dynamic", dynamic_prediction_df)]:
    missing_columns = required_columns - set(df.columns)
    assert not missing_columns, f"{name}: missing required columns: {sorted(missing_columns)}"

    assert len(df) == 756
    assert df["sample_id"].is_unique
    assert df["reenactment_id"].nunique() == 378
    assert set(df["camera_index"].unique()) == {0, 1}

    samples_per_reenactment = df.groupby("reenactment_id").size()
    assert samples_per_reenactment.eq(2).all()

    views_per_reenactment = df.groupby("reenactment_id")["camera_index"].nunique()
    assert views_per_reenactment.eq(2).all()

    true_labels_per_reenactment = df.groupby("reenactment_id")["true_label_id"].nunique()
    assert true_labels_per_reenactment.eq(1).all()

    participants_per_reenactment = df.groupby("reenactment_id")["participant_id"].nunique()
    assert participants_per_reenactment.eq(1).all()
    assert df["participant_id"].nunique() == 8

    fea_predictions_per_reenactment = df.groupby("reenactment_id")["fea_pred_id"].nunique()
    assert fea_predictions_per_reenactment.eq(1).all()

assert set(static_prediction_df["sample_id"]) == set(dynamic_prediction_df["sample_id"])
assert set(static_prediction_df["reenactment_id"]) == set(dynamic_prediction_df["reenactment_id"])

static_metadata = (
    static_prediction_df[
        ["sample_id", "reenactment_id", "participant_id", "camera_index", "true_label_id"]
    ]
    .sort_values("sample_id")
    .reset_index(drop=True)
)

dynamic_metadata = (
    dynamic_prediction_df[
        ["sample_id", "reenactment_id", "participant_id", "camera_index", "true_label_id"]
    ]
    .sort_values("sample_id")
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(static_metadata, dynamic_metadata)

print("Both prediction tables and their cross-setting alignment were validated.")

Both prediction tables and their cross-setting alignment were validated.


## 3. Construct Reenactment-Level Analysis Table

For each reenactment, let $S_i$ indicate whether the Static FEA prediction is correct and $D_i$ indicate whether the Dynamic FEA prediction is correct.

Because FEA is not view-specific, the duplicated Central and Side rows are collapsed to one prediction per reenactment before statistical testing.

The paired difference is

$d_i = D_i - S_i$

with support $\{-1, 0, 1\}$:

- $+1$: only Dynamic FEA is correct
- $-1$: only Static FEA is correct
- $0$: both models have the same correctness outcome

In [4]:
def collapse_fea_to_reenactment(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    reenactment_df = (
        df.sort_values(["reenactment_id", "camera_index"])
        .groupby("reenactment_id", as_index=False)
        .agg(
            participant_id=("participant_id", "first"),
            true_label_id=("true_label_id", "first"),
            fea_pred_id=("fea_pred_id", "first")
        )
    )

    reenactment_df[f"{prefix}_fea_correct"] = (
        reenactment_df["fea_pred_id"] == reenactment_df["true_label_id"]
    )

    return reenactment_df.rename(
        columns={
            "participant_id": f"{prefix}_participant_id",
            "true_label_id": f"{prefix}_true_label_id",
            "fea_pred_id": f"{prefix}_fea_pred_id"
        }
    )


static_fea_df = collapse_fea_to_reenactment(static_prediction_df, "static")
dynamic_fea_df = collapse_fea_to_reenactment(dynamic_prediction_df, "dynamic")

analysis_df = static_fea_df.merge(
    dynamic_fea_df,
    on="reenactment_id",
    how="inner",
    validate="one_to_one"
)

assert len(analysis_df) == 378
assert analysis_df["static_participant_id"].equals(analysis_df["dynamic_participant_id"])
assert analysis_df["static_true_label_id"].equals(analysis_df["dynamic_true_label_id"])

analysis_df["participant_id"] = analysis_df["static_participant_id"]
analysis_df["true_label_id"] = analysis_df["static_true_label_id"]

analysis_df["difference"] = (
    analysis_df["dynamic_fea_correct"].astype(int)
    - analysis_df["static_fea_correct"].astype(int)
)

analysis_df = analysis_df[
    [
        "reenactment_id",
        "participant_id",
        "true_label_id",
        "static_fea_pred_id",
        "dynamic_fea_pred_id",
        "static_fea_correct",
        "dynamic_fea_correct",
        "difference"
    ]
]

analysis_df.head()

,reenactment_id,participant_id,true_label_id,static_fea_pred_id,dynamic_fea_pred_id,static_fea_correct,dynamic_fea_correct,difference
0,1700478995850-2-1-1-0-0,1,0,4,0,False,True,1
1,1700478998549-2-1-1-1-5,1,5,5,5,True,True,0
2,1700479001137-2-1-1-2-3,1,3,3,3,True,True,0
3,1700479004312-2-1-1-3-0,1,0,0,4,True,False,-1
4,1700479005401-2-1-1-4-0,1,0,0,0,True,True,0


## 4. Verify Reported Performance

In [5]:
static_fea_accuracy = analysis_df["static_fea_correct"].mean()
dynamic_fea_accuracy = analysis_df["dynamic_fea_correct"].mean()

static_correct = int(analysis_df["static_fea_correct"].sum())
dynamic_correct = int(analysis_df["dynamic_fea_correct"].sum())

assert static_correct == 271
assert dynamic_correct == 296

print(f"Static FEA:  {static_correct}/378 = {100 * static_fea_accuracy:.2f}%")
print(f"Dynamic FEA: {dynamic_correct}/378 = {100 * dynamic_fea_accuracy:.2f}%")
print(f"Dynamic - Static difference: {100 * (dynamic_fea_accuracy - static_fea_accuracy):.2f} percentage points")

Static FEA:  271/378 = 71.69%
Dynamic FEA: 296/378 = 78.31%
Dynamic - Static difference: 6.61 percentage points


## 5. Primary Static FEA vs. Dynamic FEA Comparison

The primary estimand is

$\Delta = \mathrm{Accuracy}_{Dynamic\ FEA} - \mathrm{Accuracy}_{Static\ FEA}$.

The primary inferential test is an **exact two-sided McNemar test** over the 378 paired reenactments.

The $2\times2$ paired-correctness table is organized as:

| | Dynamic correct | Dynamic wrong |
|---|---:|---:|
| Static correct | both correct | Static only correct |
| Static wrong | Dynamic only correct | both wrong |

Only the two discordant cells contribute to McNemar's test. Positive values of $\Delta$ favor Dynamic FEA.

In [6]:
static_correct_array = analysis_df["static_fea_correct"].to_numpy(dtype=bool)
dynamic_correct_array = analysis_df["dynamic_fea_correct"].to_numpy(dtype=bool)

both_correct = int(np.sum(static_correct_array & dynamic_correct_array))
static_only_correct = int(np.sum(static_correct_array & ~dynamic_correct_array))
dynamic_only_correct = int(np.sum(~static_correct_array & dynamic_correct_array))
both_wrong = int(np.sum(~static_correct_array & ~dynamic_correct_array))

mcnemar_table = np.array([
    [both_correct, static_only_correct],
    [dynamic_only_correct, both_wrong]
])

mcnemar_result = mcnemar(mcnemar_table, exact=True, correction=False)

print("Paired correctness table:")
print(mcnemar_table)
print()
print(f"Both correct:         {both_correct}")
print(f"Static only correct:  {static_only_correct}")
print(f"Dynamic only correct: {dynamic_only_correct}")
print(f"Both wrong:           {both_wrong}")
print()
print(f"Exact two-sided McNemar p-value: {mcnemar_result.pvalue:.12f}")

Paired correctness table:
[[255  16]
 [ 41  66]]

Both correct:         255
Static only correct:  16
Dynamic only correct: 41
Both wrong:           66

Exact two-sided McNemar p-value: 0.001263559472


## 6. Paired Bootstrap Confidence Interval

McNemar's test provides the primary hypothesis test. To quantify the effect size and its uncertainty, a paired bootstrap is used for the accuracy difference $\Delta$.

The 378 reenactment-level pairs are resampled with replacement. Static and Dynamic correctness remain paired within each resampled reenactment.

The percentile interval below is a 95% bootstrap confidence interval for $\Delta$. No additional bootstrap hypothesis test is used because the exact McNemar test is the primary inferential test.

In [7]:
def paired_bootstrap_ci(
    differences: np.ndarray,
    n_bootstrap: int,
    batch_size: int,
    alpha: float,
    random_seed: int
) -> tuple[float, float]:
    rng = np.random.default_rng(random_seed)
    n = len(differences)

    bootstrap_means = np.empty(n_bootstrap, dtype=np.float64)

    for start in range(0, n_bootstrap, batch_size):
        stop = min(start + batch_size, n_bootstrap)
        current_batch_size = stop - start

        indices = rng.integers(0, n, size=(current_batch_size, n))
        bootstrap_means[start:stop] = differences[indices].mean(axis=1)

    ci_low, ci_high = np.quantile(
        bootstrap_means,
        [alpha / 2, 1 - alpha / 2]
    )

    return float(ci_low), float(ci_high)


differences = analysis_df["difference"].to_numpy(dtype=float)

ci_low, ci_high = paired_bootstrap_ci(
    differences=differences,
    n_bootstrap=N_BOOTSTRAP,
    batch_size=BOOTSTRAP_BATCH_SIZE,
    alpha=ALPHA,
    random_seed=RANDOM_SEED
)

difference_mean = differences.mean()

primary_result_df = pd.DataFrame([{
    "comparison": "Static FEA vs. Dynamic FEA",
    "n_reenactments": len(analysis_df),
    "static_fea_accuracy": static_fea_accuracy,
    "dynamic_fea_accuracy": dynamic_fea_accuracy,
    "difference_dynamic_minus_static_pp": 100 * difference_mean,
    "ci_low_pp": 100 * ci_low,
    "ci_high_pp": 100 * ci_high,
    "both_correct": both_correct,
    "static_only_correct": static_only_correct,
    "dynamic_only_correct": dynamic_only_correct,
    "both_wrong": both_wrong,
    "mcnemar_statistic": float(mcnemar_result.statistic),
    "p_value": float(mcnemar_result.pvalue)
}])

row = primary_result_df.iloc[0]

print(f"Dynamic - Static FEA accuracy difference: {row['difference_dynamic_minus_static_pp']:.2f} percentage points")
print(f"{100 * (1 - ALPHA):.0f}% paired-bootstrap CI: [{row['ci_low_pp']:.2f}, {row['ci_high_pp']:.2f}] percentage points")
print(f"Exact two-sided McNemar p-value: {row['p_value']:.12f}")

primary_result_df

Dynamic - Static FEA accuracy difference: 6.61 percentage points
95% paired-bootstrap CI: [2.91, 10.58] percentage points
Exact two-sided McNemar p-value: 0.001263559472


,comparison,n_reenactments,static_fea_accuracy,dynamic_fea_accuracy,difference_dynamic_minus_static_pp,ci_low_pp,ci_high_pp,both_correct,static_only_correct,dynamic_only_correct,both_wrong,mcnemar_statistic,p_value
0,Static FEA vs. Dynamic FEA,378,0.716931,0.783069,6.613757,2.910053,10.582011,255,16,41,66,16.0,0.001264


## 7. Sensitivity Check

As a sensitivity analysis, the same 378 paired reenactment-level differences $d_i = D_i - S_i$ are tested with a one-sample t-test against a mean of zero.

This is not a separate primary hypothesis test and is not included in a multiplicity-correction family. It only checks whether a conventional parametric test leads to the same substantive conclusion as the exact McNemar analysis.

In [8]:
sensitivity_test = ttest_1samp(differences, popmean=0.0)

sensitivity_result_df = pd.DataFrame([{
    "comparison": "Static FEA vs. Dynamic FEA",
    "n_reenactments": len(differences),
    "mean_difference_pp": 100 * differences.mean(),
    "t_statistic": float(sensitivity_test.statistic),
    "df": int(sensitivity_test.df),
    "p_value": float(sensitivity_test.pvalue)
}])

row = sensitivity_result_df.iloc[0]

print(f"Mean difference: {row['mean_difference_pp']:.2f} percentage points")
print(f"t({int(row['df'])}) = {row['t_statistic']:.3f}")
print(f"Two-sided sensitivity-check p-value: {row['p_value']:.12f}")

sensitivity_result_df

Mean difference: 6.61 percentage points
t(377) = 3.356
Two-sided sensitivity-check p-value: 0.000871236546


,comparison,n_reenactments,mean_difference_pp,t_statistic,df,p_value
0,Static FEA vs. Dynamic FEA,378,6.613757,3.355981,377,0.000871


## 8. Participant-Level Heterogeneity Check

This descriptive check examines whether the Dynamic-vs.-Static FEA difference is directionally consistent across the eight held-out participants or is mainly driven by a small number of participants.

For each participant, the table reports the number of reenactments, Static FEA accuracy, Dynamic FEA accuracy, and the difference $\mathrm{Accuracy}_{Dynamic\ FEA} - \mathrm{Accuracy}_{Static\ FEA}$.

No participant-level significance test or multiplicity correction is applied. The primary inference remains the paired reenactment-level McNemar analysis above.

In [9]:
participant_result_df = (
    analysis_df
    .groupby("participant_id", as_index=False)
    .agg(
        n_reenactments=("reenactment_id", "size"),
        static_fea_accuracy=("static_fea_correct", "mean"),
        dynamic_fea_accuracy=("dynamic_fea_correct", "mean")
    )
)

participant_result_df["difference_pp"] = 100 * (
    participant_result_df["dynamic_fea_accuracy"]
    - participant_result_df["static_fea_accuracy"]
)

n_dynamic_better = int((participant_result_df["difference_pp"] > 0).sum())
n_static_better = int((participant_result_df["difference_pp"] < 0).sum())
n_equal = int((participant_result_df["difference_pp"] == 0).sum())
median_participant_difference_pp = participant_result_df["difference_pp"].median()

print(f"Participants favoring Dynamic FEA: {n_dynamic_better}/8")
print(f"Participants favoring Static FEA:  {n_static_better}/8")
print(f"Participants tied:                  {n_equal}/8")
print(f"Median participant difference (Dynamic - Static FEA): {median_participant_difference_pp:.2f} percentage points")

participant_result_df

Participants favoring Dynamic FEA: 6/8
Participants favoring Static FEA:  2/8
Participants tied:                  0/8
Median participant difference (Dynamic - Static FEA): 6.27 percentage points


,participant_id,n_reenactments,static_fea_accuracy,dynamic_fea_accuracy,difference_pp
0,1,47,0.574468,0.723404,14.893617
1,8,54,0.666667,0.722222,5.555556
2,10,46,0.782609,0.804348,2.173913
3,13,46,0.804348,0.934783,13.043478
4,15,48,0.645833,0.625000,-2.083333
5,18,43,0.906977,0.976744,6.976744
6,23,53,0.830189,0.792453,-3.773585
7,27,41,0.512195,0.707317,19.512195


## 9. Summary and Export

The primary result compares Static and Dynamic FEA on exactly the same 378 reenactments using an exact paired McNemar test. The paired bootstrap quantifies uncertainty in the Dynamic-minus-Static accuracy difference, while the one-sample t-test provides a sensitivity check of the same reenactment-level differences. The participant-level breakdown is descriptive and is used to assess directional consistency and heterogeneity across the eight test participants.

In [10]:
summary_df = pd.DataFrame({
    "metric": [
        "Static FEA accuracy",
        "Dynamic FEA accuracy",
        "Dynamic - Static FEA difference (pp)",
        "Paired-bootstrap CI low (pp)",
        "Paired-bootstrap CI high (pp)",
        "Exact McNemar p-value",
        "Static-only correct reenactments",
        "Dynamic-only correct reenactments",
        "Sensitivity t statistic",
        "Sensitivity t-test p-value",
        "Participants favoring Dynamic FEA",
        "Participants favoring Static FEA",
        "Participants tied",
        "Median participant difference Dynamic - Static FEA (pp)"
    ],
    "value": [
        static_fea_accuracy,
        dynamic_fea_accuracy,
        100 * difference_mean,
        100 * ci_low,
        100 * ci_high,
        mcnemar_result.pvalue,
        static_only_correct,
        dynamic_only_correct,
        sensitivity_test.statistic,
        sensitivity_test.pvalue,
        n_dynamic_better,
        n_static_better,
        n_equal,
        median_participant_difference_pp
    ]
})

summary_df

,metric,value
0,Static FEA accuracy,0.716931
1,Dynamic FEA accuracy,0.783069
2,Dynamic - Static FEA difference (pp),6.613757
3,Paired-bootstrap CI low (pp),2.910053
4,Paired-bootstrap CI high (pp),10.582011
5,Exact McNemar p-value,0.001264
6,Static-only correct reenactments,16.000000
7,Dynamic-only correct reenactments,41.000000
8,Sensitivity t statistic,3.355981
9,Sensitivity t-test p-value,0.000871


In [11]:
OUTPUT_DIR = Path("statistical_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

analysis_df.to_csv(OUTPUT_DIR / "static_fea_vs_dynamic_fea_reenactment_table.csv", index=False)
primary_result_df.to_csv(OUTPUT_DIR / "static_fea_vs_dynamic_fea_primary_result.csv", index=False)
sensitivity_result_df.to_csv(OUTPUT_DIR / "static_fea_vs_dynamic_fea_sensitivity_ttest.csv", index=False)
participant_result_df.to_csv(OUTPUT_DIR / "static_fea_vs_dynamic_fea_participant_level_results.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "static_fea_vs_dynamic_fea_summary.csv", index=False)

print(f"Results written to: {OUTPUT_DIR.resolve()}")

Results written to: /workspace/repos/emohevrdb-dfer/6_discussion/significance-tests/dynamic-significance-tests/statistical_results
